In [1]:
import re
import anthropic
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)  # reads ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic()
MODEL_NAME = "claude-haiku-5-5"
pd.set_option("display.max_colwidth", None)

def get_completion(prompt, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        system=system_prompt,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in message.content if b.type == "text")

SPECIALTIES = ["Cardiovascular / Pulmonary", "Orthopedic", "Gastroenterology",
               "Neurology", "Urology", "Obstetrics / Gynecology"]
data = pd.read_csv("data/mtsamples_subset.csv")
print(len(data), "notes loaded")
print(get_completion("Say hi in five words."))

210 notes loaded
Hello there, friend! Hope you're well.


In [2]:
practice = data.sample(8, random_state=1)
example_pool = data.drop(practice.index).sample(6, random_state=2)
print(practice["medical_specialty"].value_counts())

medical_specialty
Gastroenterology              3
Urology                       2
Neurology                     1
Cardiovascular / Pulmonary    1
Orthopedic                    1
Name: count, dtype: int64


In [3]:
practice["my_label"] = practice["medical_specialty"]

In [5]:
practice = data.sample(8, random_state=1)
example_pool = data.drop(practice.index).sample(6, random_state=2)
print(practice["medical_specialty"].value_counts())

medical_specialty
Gastroenterology              3
Urology                       2
Neurology                     1
Cardiovascular / Pulmonary    1
Orthopedic                    1
Name: count, dtype: int64


In [7]:
print(practice.iloc[1]["medical_specialty"])
print(practice.iloc[1]["transcription"])

Urology
PREOPERATIVE DIAGNOSIS:,  Adenocarcinoma of the prostate.,POSTOPERATIVE DIAGNOSIS:,  Adenocarcinoma of the prostate.,TITLE OF OPERATION:,  Mini-laparotomy radical retropubic prostatectomy with bilateral pelvic lymph node dissection with Cavermap.,ANESTHESIA: , General by intubation.,Informed consent was obtained for the procedure.  The patient understands the treatment options and wishes to proceed.  He accepts the risks to include bleeding requiring transfusion, infection, sepsis, incontinence, impotence, bladder neck constricture, heart attack, stroke, pulmonary emboli, phlebitis, injury to the bladder, rectum, or ureter, etcetera.,OPERATIVE PROCEDURE IN DETAIL: , The patient was taken to the Operating Room and placed in the supine position, prepped with Betadine solution and draped in the usual sterile fashion.  A 20- French Foley catheter was inserted into the penis and into the bladder and placed to dependent drainage.  The table was then placed in minimal flexed position.

In [8]:
practice["my_label"] = practice["medical_specialty"]

In [9]:
for n in range(len(example_pool)):
    row = example_pool.iloc[n]
    print(f"===== [{n}] {row['medical_specialty']} =====")
    print(row["transcription"][:1200])
    print()

===== [0] Gastroenterology =====
PROCEDURE: , Gastroscopy.,PREOPERATIVE DIAGNOSES: , Dysphagia, possible stricture.,POSTOPERATIVE DIAGNOSIS: , Gastroparesis.,MEDICATION: , MAC.,DESCRIPTION OF PROCEDURE: , The Olympus gastroscope was introduced into the hypopharynx and passed carefully through the esophagus, stomach, and duodenum.  The hypopharynx was normal.  The esophagus had a normal upper esophageal sphincter, normal contour throughout, and a normal gastroesophageal junction viewed at 39 cm from the incisors.  There was no evidence of stricturing or extrinsic narrowing from her previous hiatal hernia repair.  There was no sign of reflux esophagitis.  On entering the gastric lumen, a large bezoar of undigested food was seen occupying much of the gastric fundus and body.  It had 2 to 3 mm diameter.  This was broken up using a scope into smaller pieces.  There was no retained gastric liquid.  The antrum appeared normal and the pylorus was patent.  The scope passed easily into the duode

In [12]:
SYSTEM_PROMPT = f"""You are a clinical documentation specialist who routes medical transcriptions.

Your job: for one transcription, identify
1. specialty: exactly one of: {", ".join(SPECIALTIES)}
2. chief_complaint: the main reason for the encounter, in a short phrase taken from the note
3. reason: one sentence citing the specific details in the note that decided the specialty

Rules:
- Use only what the note says. Do not add diagnoses or findings that are not in the note.
- Copy the specialty name exactly as written in the list above.
- If a note touches several areas, choose the specialty of the main procedure or consult performed.

Examples of the exact output format:
<examples>
<example>
CPROCEDURE: , Gastroscopy.,PREOPERATIVE DIAGNOSES: , Dysphagia, possible stricture.</record>
<answer><specialty>Cardiovascular / Pulmonary</specialty><chief_complaint>exertional chest pain</chief_complaint><reason>Exertional chest pressure, EKG changes and a plan for cardiac catheterization point to cardiology.</reason></answer>
</example>
<example>
<record>OPERATION: , Subxiphoid pericardial window.,ANESTHESIA: , General endotracheal anesthesia.</record>
<answer><specialty>Orthopedic</specialty><chief_complaint>right knee meniscus tear</chief_complaint><reason>The main procedure is a knee arthroscopy to repair a meniscus tear.</reason></answer>
</example>
<record>CT ANGIOGRAPHY CHEST WITH CONTRAST,REASON FOR EXAM:  ,Shortness of breath for two weeks and a history of pneumonia.  The patient also has a history of left lobectomy.</record>
</examples>"""

In [13]:
print(SYSTEM_PROMPT)

You are a clinical documentation specialist who routes medical transcriptions.

Your job: for one transcription, identify
1. specialty: exactly one of: Cardiovascular / Pulmonary, Orthopedic, Gastroenterology, Neurology, Urology, Obstetrics / Gynecology
2. chief_complaint: the main reason for the encounter, in a short phrase taken from the note
3. reason: one sentence citing the specific details in the note that decided the specialty

Rules:
- Use only what the note says. Do not add diagnoses or findings that are not in the note.
- Copy the specialty name exactly as written in the list above.
- If a note touches several areas, choose the specialty of the main procedure or consult performed.

Examples of the exact output format:
<examples>
<example>
CPROCEDURE: , Gastroscopy.,PREOPERATIVE DIAGNOSES: , Dysphagia, possible stricture.</record>
<answer><specialty>Cardiovascular / Pulmonary</specialty><chief_complaint>exertional chest pain</chief_complaint><reason>Exertional chest pressure, 

In [14]:
def build_prompt(note_text, think=False):
    if think:
        instruction = ("Analyze the medical transcription in the <record> tags. First, reason step by "
                       "step inside <thinking> tags: note the body system involved, the main procedure "
                       "or consult, and the reason for the encounter. Then give your final answer in an "
                       "<answer> block with <specialty>, <chief_complaint>, and <reason>, exactly as in "
                       "the examples.")
    else:
        instruction = ("Analyze the medical transcription in the <record> tags. Respond with only an "
                       "<answer> block with <specialty>, <chief_complaint>, and <reason>, exactly as in "
                       "the examples.")
    return f"<record>\n{note_text}\n</record>\n\n{instruction}"

def extract(tag, text):
    match = re.search(f"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1).strip() if match else None

In [17]:
rows = []
for _, r in practice.iterrows():
    plain = get_completion(build_prompt(r["transcription"]), SYSTEM_PROMPT)
    thought = get_completion(build_prompt(r["transcription"], think=True), SYSTEM_PROMPT)
    rows.append({
        "my_label": r["my_label"],
        "spec_plain": extract("specialty", plain),
        "spec_think": extract("specialty", thought),
        "cc_think": extract("chief_complaint", thought),
        "reason_think": extract("reason", thought),
        "thinking": extract("thinking", thought),
        "note_start": r["transcription"][:200],
    })

results = pd.DataFrame(rows)
print("Plain accuracy:   ", (results.spec_plain == results.my_label).mean())
print("Thinking accuracy:", (results.spec_think == results.my_label).mean())
results

Plain accuracy:    1.0
Thinking accuracy: 1.0


,my_label,spec_plain,spec_think,cc_think,reason_think,thinking,note_start
0,Gastroenterology,Gastroenterology,Gastroenterology,colovesical fistula with stool passing from the urethra,"The main procedure is a low anterior colon resection with colostomy and colonoscopy for perforated diverticulitis causing a colovesical fistula, while the bladder repair was secondary.","The note involves the colon, bladder, and peritoneum. The main procedures are a low anterior colon resection, colonoscopy, and a transverse loop colostomy, all for a colovesical fistula. Bladder repair was done by a second surgeon and is secondary to the colon disease. The encounter's main reason is stool passing through the urethra due to a colovesical fistula.","PREOPERATIVE DIAGNOSIS: , Colovesical fistula.,POSTOPERATIVE DIAGNOSES:,1. Colovesical fistula.,2. Intraperitoneal abscess.,PROCEDURE PERFORMED:,1. Exploratory laparotomy.,2. Low anterior colon re"
1,Urology,Urology,Urology,adenocarcinoma of the prostate,"The main procedure is a radical retropubic prostatectomy with bilateral pelvic lymph node dissection for prostate adenocarcinoma, including bladder neck reconstruction and urethral anastomosis.","- Body system: prostate and pelvis (genitourinary).\n- Main procedure: mini-laparotomy radical retropubic prostatectomy with bilateral pelvic lymph node dissection.\n- Reason for encounter: preoperative and postoperative diagnosis of adenocarcinoma of the prostate.\n- The note covers bladder neck reconstruction, urethral anastomosis, and ureteral orifice assessment, which are all part of the same urologic procedure.\n- The lymph node dissection is part of the prostate cancer surgery, not a separate specialty.","PREOPERATIVE DIAGNOSIS:, Adenocarcinoma of the prostate.,POSTOPERATIVE DIAGNOSIS:, Adenocarcinoma of the prostate.,TITLE OF OPERATION:, Mini-laparotomy radical retropubic prostatectomy with bilater"
2,Gastroenterology,Gastroenterology,Gastroenterology,"right lower quadrant abdominal pain, suspected acute appendicitis","The main procedure is a laparoscopic appendectomy for suppurative appendicitis, following right lower quadrant tenderness with rebound, leukocytosis of 12.8, and a non-diagnostic CT, so the encounter concerns a gastrointestinal organ.","- Body system: abdomen. The pain is right lower quadrant, and the organ involved is the appendix, part of the gastrointestinal tract.\n- Main procedure: laparoscopic appendectomy, after diagnostic laparoscopy. The postoperative diagnosis is acute suppurative appendicitis.\n- Reason for encounter: right lower quadrant abdominal pain with suspected acute appendicitis, supported by rebound tenderness, leukocytosis of 12.8, and a non-diagnostic CT.\n- Specialty: General Surgery is not on the approved list. Gastroenterology is the closest match because appendicitis is a gastrointestinal condition and the procedure treats the GI tract. Obstetrics/Gynecology does not fit: the ovarian follicular cysts were incidental and did not drive the procedure.","PREOPERATIVE DIAGNOSIS: , Right lower quadrant abdominal pain, rule out acute appendicitis.,POSTOPERATIVE DIAGNOSIS:, Acute suppurative appendicitis.,PROCEDURE PERFORMED:,1. Diagnostic laparoscopy.,"
3,Urology,Urology,Urology,abrupt left testicular pain with suspected testicular torsion,The main procedure is a left scrotal exploration with detorsion and bilateral testicular fixation for suspected left testicular torsion in a 4-year-old boy with abrupt onset of left testicular pain.,"The note concerns the scrotum and testes. The main procedure is a left scrotal exploration with detorsion and bilateral testicular fixation, and the presenting problem is abrupt left testicular pain with suspected torsion. Both point to urology, not obstetrics/gynecology or another listed specialty.","PREOPERATIVE DIAGNOSIS: , Left testicular torsion, possibly detorsion.,POSTOPERATIVE DIAGNOSIS: , Left testicular torsion, possibly detorsion.,PROCEDURE: , Left scrotal exploration with deto

In [18]:
print("Plain accuracy:   ", (results.spec_plain == results.my_label).mean())
print("Thinking accuracy:", (results.spec_think == results.my_label).mean())
print("Format failures:  ", results.isna().sum().sum())

misses = results[(results.spec_plain != results.my_label) | (results.spec_think != results.my_label)]
misses[["my_label", "spec_plain", "spec_think", "reason_think"]]

Plain accuracy:    1.0
Thinking accuracy: 1.0
Format failures:   0


,my_label,spec_plain,spec_think,reason_think
